# Environment

In [ ]:
import os
os.chdir("../..")

In [ ]:
import pandas as pd
import spacy

from src.utils.path import PROCESSED_DATA_PATH, RAW_LEXICONS_PATH, FEATURES_PATH

In [ ]:
nlp = spacy.load("pt_core_news_lg")

## Load SentiLex

`POL:N0` → polaridade quando o alvo humano é sujeito.

`POL:N1` → polaridade quando o alvo humano é complemento.

In [ ]:
sentilex = {}

with open(f"{RAW_LEXICONS_PATH}/SentiLex-lem-PT02.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split(";")
        word = parts[0].split(".")[0].lower()
        polarities = {}
        for p in parts:
            if p.startswith("POL:N0="):
                polarities["N0"] = int(p.split("=")[1])
            elif p.startswith("POL:N1="):
                polarities["N1"] = int(p.split("=")[1])
        sentilex[word] = polarities

# Feature Extraction Functions

## Opinion Target Finder

Identifica o alvo semântico de cada palavra com polaridade no texto.

In [ ]:
class OpinionTargetFinder:
    def __init__(self, cop_deps, subj_deps, noun_mod_deps, obj_deps):
        self.COP_DEPS = cop_deps
        self.SUBJ_DEPS = subj_deps
        self.NOUN_MOD_DEPS = noun_mod_deps
        self.OBJ_DEPS = obj_deps

    @staticmethod
    def _unique_tokens(tokens):
        seen = set()
        unique = []
        for t in tokens:
            if t.i not in seen:
                seen.add(t.i)
                unique.append(t)
        return unique

    def _targets_predicativo(self, token):
        targets = []
        if token.dep_ in self.COP_DEPS and token.head.pos_ in ("VERB", "AUX"):
            for child in token.head.children:
                if child.dep_ in self.SUBJ_DEPS:
                    targets.append(child)
        if any(c.dep_ == "cop" and c.pos_ in ("VERB", "AUX") for c in token.children):
            for child in token.children:
                if child.dep_ in self.SUBJ_DEPS:
                    targets.append(child)
        return targets

    def _get_targets_for_adj(self, token):
        targets = []
        if token.dep_ == "amod" and token.head.pos_ == "NOUN":
            targets.append(token.head)
        targets.extend(self._targets_predicativo(token))
        return self._unique_tokens(targets)

    def _get_targets_for_noun(self, token):
        targets = []
        targets.extend(self._targets_predicativo(token))
        if token.dep_ in self.NOUN_MOD_DEPS and token.head.pos_ == "NOUN":
            targets.append(token.head)
        for child in token.children:
            if child.pos_ == "ADP" and child.lemma_.lower() == "de":
                for obj in child.children:
                    if obj.pos_ == "NOUN":
                        targets.append(obj)
        if token.dep_ in self.SUBJ_DEPS:
            for child in token.children:
                if child.pos_ == "NOUN":
                    targets.append(child)
        return self._unique_tokens(targets)

    def _get_targets_for_verb(self, token):
        targets = []
        for child in token.children:
            if child.dep_ in self.OBJ_DEPS:
                targets.append(child)
            if child.dep_ == "nsubj:pass":
                targets.append(child)
        return self._unique_tokens(targets)

    def extract_target_for_token(self, token):
        if token.pos_ == "ADJ": return self._get_targets_for_adj(token)
        if token.pos_ == "NOUN": return self._get_targets_for_noun(token)
        if token.pos_ == "VERB": return self._get_targets_for_verb(token)
        return []

In [ ]:
COP_DEPS = ("acomp", "attr", "xcomp", "ROOT")
SUBJ_DEPS = ("nsubj", "nsubj:pass")
NOUN_MOD_DEPS = ("nmod", "appos", "compound", "nmod:poss")
OBJ_DEPS = ("obj", "dobj", "iobj", "obl:arg")

finder = OpinionTargetFinder(COP_DEPS, SUBJ_DEPS, NOUN_MOD_DEPS, OBJ_DEPS)

SUBJ_TAGS = ["nsubj", "csubj", "nsubj:pass"]
COMP_TAGS = ["obj", "dobj", "iobj", "obl", "ccomp", "xcomp"]

## Syntactic Polarity Classifier

In [ ]:
def get_sentilex_polarity(token, target, sentilex):
    for form in (token.lemma_.lower(), token.text.lower()):
        if form in sentilex:
            return sentilex[form].get(target, 0)
    return 0

def classify_syntactic_polarity(text, finder, sentilex):
    doc = nlp(text)
    pos_tokens, neg_tokens, nto_tokens = [], [], []
    total_score = 0

    for token in doc:
        if token.is_punct:
            continue
        lex_form = None
        if token.lemma_ in sentilex: lex_form = token.lemma_
        elif token.text in sentilex: lex_form = token.text
        if lex_form is None:
            continue
        targets = finder.extract_target_for_token(token)
        target = None
        for t in targets:
            if t.dep_ in SUBJ_TAGS: target = "N0"; break
            elif t.dep_ in COMP_TAGS: target = "N1"; break
        if target is None:
            continue
        pol = get_sentilex_polarity(token, target, sentilex)
        if pol > 0: pos_tokens.append(token.text)
        elif pol < 0: neg_tokens.append(token.text)
        else: nto_tokens.append(token.text)
        total_score += pol

    label = "1" if total_score > 0 else "-1" if total_score < 0 else "0"
    return {
        "synt_pos_tokens": pos_tokens,
        "synt_neg_tokens": neg_tokens,
        "synt_nto_tokens": nto_tokens,
        "synt_score": total_score,
        "synt_label": label,
        "synt_token_count": len(doc),
    }

# Feature Engineering in Corpora

In [ ]:
SYNT_FEAT_COLS = ["synt_score", "synt_label", "synt_token_count"]

## Examples

In [ ]:
classify_syntactic_polarity("O presidente é um abominável", finder, sentilex)

In [ ]:
classify_syntactic_polarity("O presidente é bom", finder, sentilex)

## Hate-BR

In [ ]:
hate = pd.read_csv(f"{PROCESSED_DATA_PATH}/hate-br.csv")

In [ ]:
synt_series_hate = hate["text"].apply(lambda t: classify_syntactic_polarity(t, finder, sentilex))
synt_hate = pd.DataFrame(synt_series_hate.tolist(), index=hate.index)

In [ ]:
synt_hate[SYNT_FEAT_COLS].to_csv(f"{FEATURES_PATH}/hate-br-synt.csv", index=False)

## Ited-BR

In [ ]:
ited = pd.read_csv(f"{PROCESSED_DATA_PATH}/ited-br.csv")

In [ ]:
synt_series_ited = ited["text"].apply(lambda t: classify_syntactic_polarity(t, finder, sentilex))
synt_ited = pd.DataFrame(synt_series_ited.tolist(), index=ited.index)

In [ ]:
synt_ited[SYNT_FEAT_COLS].to_csv(f"{FEATURES_PATH}/ited-br-synt.csv", index=False)